# Diagnose spurious "end of signal" cue (disrupt-only training)

When training **disrupt-only**, the model might learn to use **when the signal ends** (or position in the window) instead of true precursors. This notebook helps diagnose that.

See also: `docs/SPURIOUS_END_OF_SIGNAL_DIAGNOSIS.md`.

**Checks:**
1. **Position ablation** — For disrupt windows, bin by where disruption falls in the window; if accuracy is much better when disruption is in the 2nd half, position may be a spurious cue.
2. **Clear-shot evaluation** — If clear shots are available, run the model on them; predictions should stay ~0; if they ramp near the **end of the shot**, that suggests an end-of-signal cue.
3. **Tail masking** — Zero out the last K timesteps at test; a large drop in metric suggests the model relies on the tail.

Run from **soen_fusion_zero**; set paths and checkpoint as needed.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

nb_dir = Path(os.path.abspath(".")).resolve()
soen_fusion_zero = nb_dir if (nb_dir / "disruptcnn").is_dir() else nb_dir.parent
if (soen_fusion_zero / "disruptcnn").is_dir() and str(soen_fusion_zero) not in sys.path:
    sys.path.insert(0, str(soen_fusion_zero))
os.chdir(soen_fusion_zero)
print("Working directory:", soen_fusion_zero)

## 1. Position-of-disruption distribution (disrupt-only)

For each **disrupt** subsequence, we have `first_disrupt` (index where label becomes 1). The fraction `first_disrupt / T` is where disruption sits in the window. If most windows have disruption in the **right half**, the model can learn "right side = disrupt" without learning precursors.

In [ ]:
# Build dataset to get start_idxi, stop_idxi, disrupt_idxi per subsequence
DECIMATED_ROOT = os.environ.get("DECIMATED_ROOT", "/home/idies/workspace/Storage/yhuang2/persistent/ecei/dsrpt_decimated_pca1")
CLEAR_ROOT = os.environ.get("CLEAR_ROOT", "")
disrupt_file = Path(soen_fusion_zero) / "disruptcnn" / "shots" / "d3d_disrupt_ecei.final.txt"
norm_stats = Path(soen_fusion_zero) / "norm_stats_pca1.npz"
T_fixed = 7813  # output length after decimate_extra
step_extra = 10

try:
    from disruptcnn.dataset_original import EceiDatasetOriginal
    inner = EceiDatasetOriginal(
        root=os.environ.get("ROOT", "/home/idies/workspace/Storage/yhuang2/persistent/ecei/dsrpt"),
        disrupt_file=str(disrupt_file),
        clear_file=None,
        flattop_only=True,
        normalize=norm_stats.exists(),
        data_step=10,
        nsub=781300,
        nrecept=30010,
        decimated_root=DECIMATED_ROOT,
        clear_decimated_root=CLEAR_ROOT or None,
        norm_stats_path=str(norm_stats) if norm_stats.exists() else None,
        decimate_extra=step_extra,
    )
    # first_disrupt in output (decimated) index: (disrupt_idxi - start_idxi + 1) / step_extra
    disrupt_mask = inner.disruptedi
    first_disrupt_out = np.zeros(len(inner.start_idxi), dtype=np.float64)
    first_disrupt_out[disrupt_mask] = (
        (inner.disrupt_idxi[disrupt_mask] - inner.start_idxi[disrupt_mask] + 1) / step_extra
    )
    # Fraction of window (0 = start, 1 = end)
    frac = first_disrupt_out / T_fixed
    frac = np.clip(frac, 0, 1)
    frac_disrupt = frac[disrupt_mask]
    print(f"Disrupt subsequences: {disrupt_mask.sum()}")
    print(f"first_disrupt fraction (0=start, 1=end): min={frac_disrupt.min():.3f}, max={frac_disrupt.max():.3f}, mean={frac_disrupt.mean():.3f}")
    plt.figure(figsize=(8, 3))
    plt.hist(frac_disrupt, bins=20, color="#c62828", alpha=0.7, edgecolor="black")
    plt.axvline(0.5, color="black", linestyle="--", label="50%")
    plt.xlabel("Disruption position in window (0=start, 1=end)")
    plt.ylabel("Count")
    plt.title("If most mass is right of 0.5, model can learn 'right side = disrupt' (spurious)")
    plt.legend()
    plt.tight_layout()
    plt.show()
except Exception as e:
    print("Dataset init failed (need shot list + H5):", e)
    frac_disrupt = None

## 2. Position ablation (accuracy by bin)

If you have a trained model and a validation set, evaluate **per bin** of disruption position. If accuracy is high only when disruption is in the **last 20–50%** of the window, the model is likely using position as a cue. Run this cell only if you have a checkpoint and the eval pipeline.

In [ ]:
# Placeholder: load checkpoint, run evaluate per (frac_bin), report metric per bin.
# CHECKPOINT = Path("checkpoints_tcn_ddp/best.pt")  # set to your best checkpoint
# from train_tcn_ddp_original import build_model, evaluate  # and get dataset, etc.
# bins = [(0, 0.2), (0.2, 0.4), (0.4, 0.6), (0.6, 0.8), (0.8, 1.0)]
# for lo, hi in bins:
#     inds = (frac >= lo) & (frac < hi) & disrupt_mask
#     if inds.sum() < 10: continue
#     metric = evaluate(model, dataset, inds, ...)
#     print(f"frac [{lo:.1f},{hi:.1f}]: n={inds.sum()}, F1={metric}")
print("To run position ablation: load your checkpoint, filter val indices by frac bin, evaluate per bin.")

## 3. Clear-shot evaluation

If you have **clear** (non-disruptive) shots and a trained model:
- Run the model on windows from clear shots (labels should be all 0).
- Plot predictions along the shot (e.g. last window that ends at shot end).
- **Spurious:** predictions ramp up or spike **near the end of the shot** (where there is no disruption, just end of data).

In [ ]:
print("Clear-shot eval: add clear shots to dataset (--clear-decimated-root + clear shot list),")
print("then run inference on clear-shot windows and plot prediction vs time.")

## 4. Mitigation: train with clear shots

To reduce the spurious cue, include **clear** (non-disruptive) shots in training so the model sees many windows with **all-zero** labels. Then "end of signal" is not a reliable indicator. Set `CLEAR_ROOT` and pass `--clear-decimated-root` and a clear shot list when training.